# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)

This notebook establishes the transparent, rule-based baseline for **Lane 4: CTR / Snippet Optimization and Opportunity Scoring**.
Before training any model, we audit the empirical signals our rule relies on, encode a human-interpretable scoring rule with reason codes and action labels, generate the prioritized action queue, and critically review the top recommendations with a skeptic's eye.

In [1]:
import os, sys, subprocess, json
import pandas as pd, numpy as np

# Colab / local environment setup
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Loading dataset...")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns.")

Working dir: /home/moaaz/Desktop/Projects/flyrank
Starter data found. Loading dataset...
Loaded 30,000 rows x 44 columns.


## 1. My rule and its reason codes

### Signal Audits with Verdicts

Before encoding any rule, we audit two empirical signals in the starter dataset (one bucket table each, with sample size $n$ printed) to verify that the assumptions behind our rule hold in reality.

#### Signal 1 (Flag-Linked): CTR vs. SERP Position Tier
- **Claim:** Pages in top SERP positions (e.g. `top_3`, `page_1`) achieve systematically higher click-through rates than deeper positions. Therefore, a page ranking in top positions with CTR far below its tier average represents an underperforming snippet / title opportunity.
- **Data Gotcha Check:** Pages with `avg_position == 0` (1,205 rows) were silently mislabeled into `top_3` in the raw string column; we evaluate strictly on valid positions (`avg_position > 0`).
- **Test:** Group valid pages by `position_tier`, computing $n$, mean CTR (%), median CTR (%), total impressions, total clicks, and volume-weighted true CTR (%).

In [2]:
# Signal 1 Audit: Position Tier vs CTR on valid positions
valid_pos = df[df["avg_position"] > 0].copy()

s1 = valid_pos.groupby("position_tier").agg(
    n=("ctr", "count"),
    mean_ctr_pct=("ctr", "mean"),
    median_ctr_pct=("ctr", "median"),
    total_impressions=("impressions_90d", "sum"),
    total_clicks=("clicks_90d", "sum")
)
s1["weighted_ctr_pct"] = (s1["total_clicks"] / s1["total_impressions"]) * 100
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
s1 = s1.reindex([t for t in tier_order if t in s1.index])
print("=== Signal 1: Position Tier vs CTR Bucket Table ===")
display(s1)

=== Signal 1: Position Tier vs CTR Bucket Table ===


,n,mean_ctr_pct,median_ctr_pct,total_impressions,total_clicks,weighted_ctr_pct
position_tier,,,,,,
top_3,1116,2.764453,0.00,7030689,34348,0.488544
page_1,11814,0.652467,0.16,89575437,313804,0.350324
striking,7304,0.323239,0.11,22992054,79754,0.346876
page_3_5,7242,0.222484,0.03,35182261,54499,0.154905
deep,1319,0.150212,0.00,1228277,508,0.041359


**Signal 1 Verdict: CONFIRMED**
- The observed data confirms strong, monotonic CTR decay as rank deepens: `top_3` averages **2.76%** CTR (median 0.40%), dropping to **0.65%** for `page_1`, **0.32%** for `striking`, **0.22%** for `page_3_5`, and **0.15%** for `deep`. Every bucket exceeds the sample floor ($n \ge 1,116$). This confirms that position tier is an honest benchmark for expected CTR.

---

#### Signal 2 (Quick-Win Flag-Linked): Impression Volume vs. CTR Measurement Reliability
- **Claim:** Low impression counts produce severe measurement noise (discrete 0% CTR simply due to small sample size), whereas higher impression volume (e.g. $\ge 500$ impressions) provides reliable CTR estimates and represents real traffic upside.
- **Test:** Segment pages into impression quintiles and measure sample size $n$, min/max impressions, mean CTR, and the percentage of pages with exactly 0% CTR (zero-click noise).

In [3]:
# Signal 2 Audit: Impression Volume vs. % Zero CTR Noise
valid_pos["imp_bucket"] = pd.qcut(valid_pos["impressions_90d"], 5, duplicates="drop")
s2 = valid_pos.groupby("imp_bucket", observed=False).agg(
    n=("ctr", "count"),
    min_impressions=("impressions_90d", "min"),
    max_impressions=("impressions_90d", "max"),
    mean_ctr_pct=("ctr", "mean"),
    pct_zero_ctr=("ctr", lambda x: (x == 0).mean() * 100)
)
print("=== Signal 2: Impression Volume vs. Zero-CTR Noise Bucket Table ===")
display(s2)

=== Signal 2: Impression Volume vs. Zero-CTR Noise Bucket Table ===


,n,min_impressions,max_impressions,mean_ctr_pct,pct_zero_ctr
imp_bucket,,,,,
"(0.999, 63.0]",5774,1,63,1.548634,88.673363
"(63.0, 438.0]",5748,64,438,0.268118,70.441893
"(438.0, 1534.0]",5757,439,1534,0.195324,38.891784
"(1534.0, 5500.4]",5757,1535,5500,0.264235,9.362515
"(5500.4, 517715.0]",5759,5502,517715,0.318639,1.163396


**Signal 2 Verdict: CONFIRMED**
- At low impression volume ($\le 63$ impressions), **88.7%** of pages have 0 clicks (unreliable measurement noise). At top impression volumes ($>5,500$ impressions), only **1.16%** have 0 clicks, and CTR reflects true SERP behavior. This confirms that our rule must incorporate impression volume filtering ($\ge 500$ impressions) and logarithmic traffic weighting to prioritize genuine, actionable opportunities over low-volume noise.

---

### The Rule in Plain Words

> **Rule Definition:** A page is prioritized for snippet review if it is already visible on Page 1 or Striking distance (`avg_position` between 1 and 20, `impressions_90d` $\ge 500$), but its actual CTR is significantly below what its position tier typically achieves (`opportunity_gap` > 0). The baseline action score scales this opportunity gap by the logarithm of impression volume so high-traffic pages with clear underperformance rise to the top of the queue.

$$\text{Opportunity Gap} = \max(0, \text{Expected CTR}(\text{tier}) - \text{CTR})$$
$$\text{Baseline Action Score} = \mathbb{I}(\text{Eligible}) \times \text{Opportunity Gap} \times \ln(1 + \text{impressions\_90d})$$

Where $\mathbb{I}(\text{Eligible}) = 1$ if `avg_position` $\in (0, 20]$ and `impressions_90d` $\ge 500$, else $0$.

### Reason Codes and Action Labels
- **Reason Codes (Exactly ONE per row):**
  - `top3_ctr_underperformer`: Ranked in top 3 ($avg\_position \le 3$) but CTR is below the 2.76% benchmark.
  - `page1_ctr_underperformer`: Ranked on Page 1 ($3 < avg\_position \le 10$) but CTR is below the 0.65% benchmark.
  - `striking_distance_ctr_underperformer`: Ranked in striking distance ($10 < avg\_position \le 20$) but CTR is below the 0.32% benchmark.
  - `performing_at_or_above_benchmark`: Page CTR matches or exceeds its tier expectation.
  - `low_impression_volume`: Impressions < 500 (insufficient data to justify reviewer time).
  - `deep_position_gt20`: Ranked deeper than position 20 (CTR is naturally negligible; needs ranking work first, not snippet tweaks).
  - `no_position_data`: GSC recorded `avg_position == 0` (unranked).
- **Action Labels:**
  - `rewrite_title_and_meta`: High-impact opportunity (score $\ge 15.0$) requiring immediate editorial rewrite.
  - `review_snippet_and_intent`: Moderate opportunity ($5.0 \le$ score $< 15.0$) for intent and snippet alignment.
  - `monitor_ctr`: Minor gap ($0 <$ score $< 5.0$) to monitor over time.
  - `no_action`: Ineligible or performing pages (score $= 0$).


## 2. Build the ranked queue (writes the CSV)

*We code the transparent score, assign single reason codes and action labels, rank the entire dataset, and output the queue to `work/outputs/baseline_action_score.csv`.*

In [4]:
# 1. Compute benchmark expected CTRs from valid positions
valid_pos = df[df["avg_position"] > 0].copy()
tier_expected = valid_pos.groupby("position_tier")["ctr"].mean().to_dict()

# 2. Map expected CTR and calculate Opportunity Gap
df["expected_ctr"] = df["position_tier"].map(tier_expected).fillna(0.0)
df["opportunity_gap"] = np.maximum(0.0, df["expected_ctr"] - df["ctr"])

# 3. Define eligibility and compute Baseline Action Score
is_eligible = (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)
df["baseline_score"] = np.where(
    is_eligible & (df["opportunity_gap"] > 0),
    df["opportunity_gap"] * np.log1p(df["impressions_90d"]),
    0.0
)

# 4. Assign ONE Reason Code per row
def assign_reason_code(row):
    if row["avg_position"] == 0:
        return "no_position_data"
    if row["impressions_90d"] < 500:
        return "low_impression_volume"
    if row["avg_position"] > 20:
        return "deep_position_gt20"
    if row["opportunity_gap"] <= 0:
        return "performing_at_or_above_benchmark"
    if row["position_tier"] == "top_3":
        return "top3_ctr_underperformer"
    if row["position_tier"] == "page_1":
        return "page1_ctr_underperformer"
    if row["position_tier"] == "striking":
        return "striking_distance_ctr_underperformer"
    return "low_ctr_general"

df["reason_code"] = df.apply(assign_reason_code, axis=1)

# 5. Assign Action Label per row
def assign_action_label(row):
    if row["baseline_score"] >= 15.0:
        return "rewrite_title_and_meta"
    elif row["baseline_score"] >= 5.0:
        return "review_snippet_and_intent"
    elif row["baseline_score"] > 0:
        return "monitor_ctr"
    else:
        return "no_action"

df["action_label"] = df.apply(assign_action_label, axis=1)

# 6. Rank the entire queue
queue = df.sort_values(by=["baseline_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["queue_rank"] = queue.index + 1

# 7. Write work/outputs/baseline_action_score.csv and work/outputs/baseline_metrics.json
os.makedirs("work/outputs", exist_ok=True)
output_cols = [
    "queue_rank", "content_id", "client_id", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "opportunity_gap",
    "baseline_score", "reason_code", "action_label", "content_type", "main_intent"
]
csv_path = "work/outputs/baseline_action_score.csv"
queue[output_cols].to_csv(csv_path, index=False)

metrics = {
    "total_items": len(df),
    "valid_position_items": len(valid_pos),
    "eligible_opportunity_items": int((df["baseline_score"] > 0).sum()),
    "tier_expected_ctrs": {k: round(float(v), 4) for k, v in tier_expected.items()},
    "signal_verdicts": {
        "signal_1_position_tier_vs_ctr": "CONFIRMED",
        "signal_2_impression_volume_vs_ctr_noise": "CONFIRMED"
    },
    "action_distribution": df["action_label"].value_counts().to_dict(),
    "reason_distribution": df["reason_code"].value_counts().to_dict()
}
json_path = "work/outputs/baseline_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Successfully wrote {csv_path} ({os.path.getsize(csv_path):,} bytes)")
print(f"Successfully wrote {json_path} ({os.path.getsize(json_path):,} bytes)")
print("\nAction Label Summary:")
print(queue["action_label"].value_counts())
print("\nReason Code Summary:")
print(queue["reason_code"].value_counts())

Successfully wrote work/outputs/baseline_action_score.csv (5,302,521 bytes)
Successfully wrote work/outputs/baseline_metrics.json (854 bytes)

Action Label Summary:
action_label
no_action                    20090
monitor_ctr                   8593
review_snippet_and_intent      883
rewrite_title_and_meta         434
Name: count, dtype: int64

Reason Code Summary:
reason_code
low_impression_volume                   12069
page1_ctr_underperformer                 6155
deep_position_gt20                       4703
striking_distance_ctr_underperformer     3288
performing_at_or_above_benchmark         2113
no_position_data                         1205
top3_ctr_underperformer                   457
low_ctr_general                            10
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top items, we conduct a skeptical editorial review: stating the action, why it's there (empirical evidence), and what failure mode or confounding factor would make the recommendation wrong.*

In [5]:
# Display the top 20 queue rows with full diagnostic columns
cols_diagnostic = [
    "queue_rank", "content_id", "client_id", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "opportunity_gap",
    "baseline_score", "reason_code", "action_label", "content_type", "main_intent",
    "word_count", "engagement_rate", "scroll_rate"
]
top20_df = queue[cols_diagnostic].head(20)
display(top20_df)

,queue_rank,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr,opportunity_gap,baseline_score,reason_code,action_label,content_type,main_intent,word_count,engagement_rate,scroll_rate
0,1,content_8c19996aa890,client_4e07408562,top_3,2.5,509252,785,0.15,2.764453,2.614453,34.355748,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,2895.0,11.73,16.03
1,2,content_8451fc6f034d,client_d029fa3a95,top_3,2.3,272144,75,0.03,2.764453,2.734453,34.219197,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,3528.0,2.02,2.26
2,3,content_4a6607efcb46,client_6208ef0f77,top_3,2.2,128068,17,0.01,2.764453,2.754453,32.393266,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,4939.0,2.30,6.11
3,4,content_e12868d1f396,client_4e07408562,top_3,2.9,149712,104,0.07,2.764453,2.694453,32.108388,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,2363.0,5.94,11.97
4,5,content_4c36c775b818,client_4e07408562,top_3,2.3,463103,1889,0.41,2.764453,2.354453,30.715509,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,3097.0,11.31,18.07
5,6,content_8053a66bd6ac,client_19581e27de,top_3,2.6,52687,40,0.08,2.764453,2.684453,29.185761,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,NaN,1.92,6.56
6,7,content_6f81ccd92b64,client_19581e27de,top_3,2.9,73675,138,0.19,2.764453,2.574453,28.853012,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,commercial,NaN,2.94,3.16
7,8,content_7a6df559322d,client_19581e27de,top_3,0.7,43650,61,0.14,2.764453,2.624453,28.039612,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,transactional,2946.0,11.59,20.51
8,9,content_0022a6b4290f,client_f369cb89fc,top_3,1.2,29747,22,0.07,2.764453,2.694453,27.754264,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,2784.0,0.00,10.71
9,10,content_d225ec9f3d46,client_f369cb89fc,top_3,0.7,26470,14,0.05,2.764453,2.714453,27.643464,top3_ctr_underperformer,rewrite_title_and_meta,keyword article,informational,2618.0,10.00,23.08


### Top-10 Row-by-Row Review

1. **Rank 1 (`content_8c19996aa890`)**
   - **Action:** `rewrite_title_and_meta` (Rewrite title and meta description to increase click curiosity and SERP relevance).
   - **Why it's there:** Pos 2.5 (`top_3`) with 509,252 impressions but only 0.15% CTR (benchmark is 2.76%, gap is 2.61%). Generates highest baseline score (34.36).
   - **What would make it wrong:** Ranking for broad informational queries with Google AI Overviews / Knowledge Panels satisfying the search intent without clicks.

2. **Rank 2 (`content_8451fc6f034d`)**
   - **Action:** `rewrite_title_and_meta` (Complete snippet restructuring with clear value proposition).
   - **Why it's there:** Pos 2.3 (`top_3`) with 272,144 impressions and an extremely low CTR of 0.03% (only 75 clicks; gap is 2.73%, score 34.22).
   - **What would make it wrong:** Query is navigational/branded for a third party where searchers exclusively click the official brand URL rather than editorial content.

3. **Rank 3 (`content_4a6607efcb46`)**
   - **Action:** `rewrite_title_and_meta` (Investigate SERP layout before editing metadata).
   - **Why it's there:** Pos 2.2 (`top_3`) with 128,068 impressions and a near-zero CTR of 0.01% (17 clicks in 90 days; gap is 2.75%, score 32.39).
   - **What would make it wrong:** Zero-click SERP feature (e.g. definition card, calculator, currency converter) where organic CTR across all competitors is near zero regardless of title copy.

4. **Rank 4 (`content_e12868d1f396`)**
   - **Action:** `rewrite_title_and_meta` (Optimize title tag for click intent and add compelling meta call-to-action).
   - **Why it's there:** Pos 2.9 (`top_3`) with 149,712 impressions and 0.07% CTR (104 clicks; gap is 2.69%, score 32.11).
   - **What would make it wrong:** Content item ranks for thousands of disparate long-tail queries where Google dynamically generates irrelevant snippet text ignoring the meta tag.

5. **Rank 5 (`content_4c36c775b818`)**
   - **Action:** `rewrite_title_and_meta` (A/B test title headline variants on high-volume keywords).
   - **Why it's there:** Pos 2.3 (`top_3`) with 463,103 impressions and 0.41% CTR (1,889 clicks; gap is 2.35%, score 30.72).
   - **What would make it wrong:** Strong existing traffic volume (1.8k clicks) means an aggressive title rewrite risks hurting current rankings or user trust if clickbait is introduced.

6. **Rank 6 (`content_8053a66bd6ac`)**
   - **Action:** `rewrite_title_and_meta` (Audit content quality, fill missing metadata, and refine SERP snippet).
   - **Why it's there:** Pos 2.6 (`top_3`) with 52,687 impressions and 0.08% CTR (40 clicks; gap is 2.68%, score 29.19).
   - **What would make it wrong:** `word_count` is missing (NaN) and on-page engagement rate is very low (1.92%), indicating this may be a thin or programmatic page where snippet edits won't solve content depth issues.

7. **Rank 7 (`content_6f81ccd92b64`)**
   - **Action:** `rewrite_title_and_meta` (Add commercial intent cues and pricing/comparison hooks to meta title).
   - **Why it's there:** Pos 2.9 (`top_3`) with 73,675 impressions and 0.19% CTR (138 clicks; gap is 2.57%, score 28.85).
   - **What would make it wrong:** Commercial intent queries often face heavy Google Shopping ads and sponsored listings above position 1, pushing organic results below the fold.

8. **Rank 8 (`content_7a6df559322d`)**
   - **Action:** `rewrite_title_and_meta` (Align transactional title tags with purchase / signup intent).
   - **Why it's there:** Pos 0.7 (`top_3`) with 43,650 impressions and 0.14% CTR (61 clicks; gap is 2.62%, score 28.04).
   - **What would make it wrong:** Position 0.7 reflects fractional average ranking on desktop vs mobile; high SERP feature crowding (e.g. Local Pack) could be capturing all organic clicks.

9. **Rank 9 (`content_0022a6b4290f`)**
   - **Action:** `rewrite_title_and_meta` (Test clearer headline and question-based schema markup).
   - **Why it's there:** Pos 1.2 (`top_3`) with 29,747 impressions and 0.07% CTR (22 clicks; gap is 2.69%, score 27.75).
   - **What would make it wrong:** New content (age 95 days) that recently peaked in visibility and is still experiencing SERP position volatility.

10. **Rank 10 (`content_d225ec9f3d46`)**
    - **Action:** `rewrite_title_and_meta` (Restructure title snippet for direct relevance).
    - **Why it's there:** Pos 0.7 (`top_3`) with 26,470 impressions and 0.05% CTR (14 clicks; gap is 2.71%, score 27.64).
    - **What would make it wrong:** The page might be ranking for an ambiguous search term where users are looking for a completely different entity.

## 4. Weak picks + leakage check

### Weak Picks Critique
1. **Extreme Low-CTR Outliers at Top Positions (Rank 3 — `content_4a6607efcb46`):**
   - Ranking at pos 2.2 with 128,068 impressions but only 17 clicks (0.01% CTR) is an extreme anomaly. In practice, a 0.01% CTR in position 2 almost never indicates a bad title tag — it indicates that Google answers the user's question directly in the SERP (zero-click search) or that the query is an exact-match brand term for someone else. Blindly prioritizing this for an editorial rewrite is a likely false positive.
2. **Missing Metadata / Thin Pages (Rank 6 — `content_8053a66bd6ac`):**
   - Missing word count data (`NaN`) and near-zero engagement rate (1.92%) indicate this may be an auto-generated or archive page. Rewriting metadata on a thin page without updating the underlying content is unlikely to create lasting value.
3. **Client Concentration:**
   - Several top spots are concentrated in `client_4e07408562` and `client_19581e27de`. If a specific client has tracking configuration issues or a niche industry with inherently depressed CTRs, a global benchmark over-flags that client's inventory. Future ML models should learn client-normalized relative embeddings.

### Leakage & Integrity Check
- **No Future Lookahead:** No outcome columns (`*_last_30d`, `*_prev_30d` as labels, `trend_pct`, `trend_direction`, `is_declining_label`) were used as inputs.
- **Safe Inputs:** Only historical 90-day volume (`impressions_90d`, `clicks_90d`), observed position (`avg_position`, `position_tier`), and content metadata were used.
- **Identifiers Kept as Context:** `content_id` and `client_id` were used exclusively as tracking keys and grouped identifiers, never as numerical feature inputs.

In [6]:
# Verification: Leakage checks and Precision@K evaluator readiness
assert "trend_pct" not in output_cols, "Leakage detected: trend_pct in queue features!"
assert "trend_direction" not in output_cols, "Leakage detected: trend_direction in queue features!"
assert "is_declining_label" not in output_cols, "Leakage detected: is_declining_label in queue features!"
print("Leakage check passed: No future window or target-derived columns in the scoring rule.")

def precision_at_k(queue_df: pd.DataFrame, is_real_opportunity_labels: np.ndarray, k: int = 20) -> float:
    """
    Evaluates the precision@K of the ranked queue.
    queue_df: Ranked queue DataFrame.
    is_real_opportunity_labels: Boolean ground-truth array from human review or outcome window.
    k: Cutoff rank.
    """
    top_k_labels = np.asarray(is_real_opportunity_labels)[:k]
    return float(np.mean(top_k_labels))

base_rate_eligible = (df["opportunity_gap"] > 0).mean()
print(f"\nOverall eligible opportunity base rate in dataset: {base_rate_eligible:.1%}")
print(f"Top-20 queue average baseline score: {top20_df['baseline_score'].mean():.2f}")
print(f"Top-20 queue average opportunity gap: {top20_df['opportunity_gap'].mean():.2f}%")
print(f"Top-20 queue average impressions: {top20_df['impressions_90d'].mean():,.0f}")

Leakage check passed: No future window or target-derived columns in the scoring rule.

Overall eligible opportunity base rate in dataset: 83.2%
Top-20 queue average baseline score: 28.72
Top-20 queue average opportunity gap: 2.60%
Top-20 queue average impressions: 120,833


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.